In [27]:
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn as nn
import torch.optim as optim
import torch
import wandb

len(os.listdir('train/ants'))

124

In [28]:
len(os.listdir('train/bees'))

121

### Transformations

In [29]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

### Load data

In [30]:
train_data = datasets.ImageFolder('train', transform=train_transforms)
val_data = datasets.ImageFolder('val', transform=val_transforms)
print(train_data.classes)

['ants', 'bees']


### DataLoaders

In [31]:
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
val_loader = DataLoader(val_data, batch_size=8, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([8, 3, 224, 224]) torch.Size([8])


### ResNet18, Freezing backbone

In [32]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


### Head Change

In [33]:
model.fc = nn.Linear(512, 2)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

fc.weight
fc.bias


### Train Loop

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(lr):
    wandb.init(
        project="TransferLearning",
        name=f"lr: {lr}",
        config={'lr': lr, 'batch_size': 8, 'epochs': 20},
    )

    model.to(device)
    optimizer = optim.Adam(model.fc.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(20):
        model.train()
        running_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        avg_loss = running_loss / len(train_loader)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                output = model(X)
                prediction = output.argmax(dim=1)
                correct += (prediction == y).sum().item()
                total += len(y)
        accuracy = correct / total
        wandb.log({"loss": avg_loss, "Accuracy": accuracy, "epoch": epoch})
    wandb.finish()

train_model(lr=0.001)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\kubak\_netrc.
wandb: Currently logged in as: jacob_21 (jacob_21-agh-univeristy-of-krakow) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run aw2qaqeo
wandb: Tracking run with wandb version 0.28.2
wandb: Run data is saved locally in C:\Users\kubak\Documents\Programming\PycharmProjects\MachineLearning\TransferLearning\wandb\run-20260905_160522-aw2qaqeo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lr: 0.001
wandb:  View project at https://wandb.ai/jacob_21-agh-univeristy-of-krakow/TransferLearning
wandb:  View run at https://wandb.ai/jacob_21-agh-univeristy-of-krakow/TransferLearning/runs/aw2qaqeo
wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: Accuracy ▁▄▅▅▅▇▇██▆▆▆▇█▇▂▆▅▇▇
wandb:    epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:     loss █▄▃▃▁▁▂▂▂▂▂▂▃▂▃▂▁▁▂▁
wandb: 
wandb: Run summary:
wandb: Accuracy 0.94771
wandb:    epoch 19
wandb:     loss

### Model trained from zero

In [35]:
model_scratch = resnet18(weights=None)
model_scratch.fc = nn.Linear(512, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(lr):
    wandb.init(
        project="TransferLearning",
        name=f"scratch-lr: {lr}",
        config={'lr': lr, 'batch_size': 8, 'epochs': 20},
    )

    model_scratch.to(device)
    optimizer = optim.Adam(model_scratch.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(20):
        model_scratch.train()
        running_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            output = model_scratch(X)
            loss = criterion(output, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        avg_loss = running_loss / len(train_loader)

        model_scratch.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                output = model_scratch(X)
                prediction = output.argmax(dim=1)
                correct += (prediction == y).sum().item()
                total += len(y)
        accuracy = correct / total
        wandb.log({"loss": avg_loss, "Accuracy": accuracy, "epoch": epoch})
    wandb.finish()

train_model(lr=0.001)

wandb: setting up run n0dvvnz6
wandb: Tracking run with wandb version 0.28.2
wandb: Run data is saved locally in C:\Users\kubak\Documents\Programming\PycharmProjects\MachineLearning\TransferLearning\wandb\run-20260905_162139-n0dvvnz6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch-lr: 0.001
wandb:  View project at https://wandb.ai/jacob_21-agh-univeristy-of-krakow/TransferLearning
wandb:  View run at https://wandb.ai/jacob_21-agh-univeristy-of-krakow/TransferLearning/runs/n0dvvnz6
wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: Accuracy ▁▁▅▅▁▆▆▄▃▇▇▄▄▇▃██▆▆▆
wandb:    epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:     loss █▃▃▃▂▂▂▂▂▃▂▂▂▂▃▂▁▁▂▂
wandb: 
wandb: Run summary:
wandb: Accuracy 0.66667
wandb:    epoch 19
wandb:     loss 0.60001
wandb: 
wandb:  View run scratch-lr: 0.001 at: https://wandb.ai/jacob_21-agh-univeristy-of-krakow/TransferLearning/runs/n0dvvnz6
wandb:  View project at: https://wandb.ai/jacob_21-agh-univeristy-of-krakow/TransferLearning
wa